In [2]:
import pandas as pd
import numpy as np
import re
from xlsxwriter.utility import xl_col_to_name

In [3]:
areas = 'PRUDENCE'
variable = 'tas95'

In [4]:
import pandas as pd
import numpy as np
import re
from xlsxwriter.utility import xl_col_to_name, xl_rowcol_to_cell

# ============================================================
# USER PARAMETERS
# ============================================================
input_csv = f"{areas}_{variable}_1989-2008_summary_results.csv"
output_excel = f"{areas}_{variable}_1989-2008_summary_results_analysis.xlsx"
output_csv = f"{areas}_{variable}_1989-2008_summary_results_analysis.csv"

# ============================================================
# DATA PROCESSING
# ============================================================
df = pd.read_csv(input_csv, header=[0, 1])

def split_values(val):
    """Parse 'MAE(STD)' strings into numeric values."""
    if pd.isna(val) or val == "" or str(val).strip() == "0":
        return pd.Series([None, None])
    match = re.match(r"\s*([0-9.]+)\s*\(\s*([0-9.]+)\s*\)", str(val))
    if match:
        v1 = float(match.group(1))
        v2 = float(match.group(2))
        return pd.Series([v1, v2])
    return pd.Series([None, None])

index_col = df.iloc[:, 0].astype(str).str.strip()
data_dict = {}

for col in df.columns[1:]:
    season, project = col
    mae_vals, std_vals = zip(*df[col].apply(split_values).values)
    data_dict[(f"{season}_MAE", project)] = mae_vals
    data_dict[(f"{season}_STD", project)] = std_vals

df_out = pd.DataFrame(data_dict, index=index_col)
df_out = df_out.dropna(how='all')
df_out.index.name = None

# ============================================================
# AUTOMATIC THRESHOLD CALCULATION
# ============================================================
def compute_thresholds(metric_keyword):
    diffs = []
    # Buscamos columnas del mismo tipo (MAE o STD)
    cols_mae_or_std = [c for c in df_out.columns if metric_keyword in c[0]]
    
    # Comparamos por pares CMIP5 y CMIP6 dentro de esa métrica
    for i in range(0, len(cols_mae_or_std), 2):
        if i+1 < len(cols_mae_or_std):
            cmip5 = df_out[cols_mae_or_std[i]]
            cmip6 = df_out[cols_mae_or_std[i+1]]
            diff = (cmip6 - cmip5).dropna()
            diffs.extend(diff.values)

    if not diffs: return 0.1, 1.0
    abs_diffs = np.abs(diffs)
    slight = max(np.percentile(abs_diffs, 25), 0.05)
    strong = max(np.percentile(abs_diffs, 75), 0.1)
    return slight, strong

mae_slight, mae_strong = compute_thresholds("MAE")
std_slight, std_strong = compute_thresholds("STD")

# ============================================================
# EXCEL EXPORT
# ============================================================
with pd.ExcelWriter(output_excel, engine="xlsxwriter") as writer:
    df_out.to_excel(writer, sheet_name="Analysis", startrow=2, header=False)
    workbook = writer.book
    worksheet = writer.sheets["Analysis"]

    # FORMATOS
    header_format = workbook.add_format({'bold': True, 'border': 1, 'align': 'center', 'bg_color': '#F2F2F2'})
    formats = {
        "dark_red": workbook.add_format({'bg_color':'#8B0000','font_color':'#FFFFFF','border':1,'align':'center'}),
        "red": workbook.add_format({'bg_color':'#F8696B','border':1,'align':'center'}),
        "orange": workbook.add_format({'bg_color':'#FDC08A','border':1,'align':'center'}),
        "white": workbook.add_format({'bg_color':'#FFFFFF','border':1,'align':'center'}),
        "very_light_green": workbook.add_format({'bg_color':'#E8F5E9','border':1,'align':'center'}),
        "green": workbook.add_format({'bg_color':'#63BE7B','border':1,'align':'center'}),
        "dark_green": workbook.add_format({'bg_color':'#006100','font_color':'#FFFFFF','border':1,'align':'center'}),
        "border": workbook.add_format({'border':1,'align':'center'})
    }

    # HEADERS
    worksheet.merge_range(0, 0, 1, 0, "Project", header_format)
    for i, col in enumerate(df_out.columns):
        worksheet.write(0, i + 1, col[0], header_format)
        worksheet.write(1, i + 1, col[1], header_format)

    # RANGOS (Ajustado para acabar una fila más abajo)
    data_start_row = 2 # Fila 3 en Excel
    data_end_row = data_start_row + len(df_out) # +1 fila extra respecto al dataframe

    for i, col in enumerate(df_out.columns):
        excel_col = i + 1 # Columna actual en Excel
        
        if "CMIP6" in col[ project_idx := 1]:
            # SEGÚN TU IMAGEN: MAE(5) STD(5) MAE(6) STD(6)
            # El CMIP5 correspondiente está 2 columnas a la izquierda
            cmip6_col_letter = xl_col_to_name(excel_col)
            cmip5_col_letter = xl_col_to_name(excel_col - 2) 
            
            # Umbrales
            slight, strong = (mae_slight, mae_strong) if "MAE" in col[0] else (std_slight, std_strong)
            
            # Fórmulas relativas (sin $) empezando en fila 3
            diff_f = f"({cmip6_col_letter}3-{cmip5_col_letter}3)"
            is_num = f"AND(ISNUMBER({cmip6_col_letter}3),ISNUMBER({cmip5_col_letter}3))"

            # 1. EMPEORA MUY FUERTE (Rojo Oscuro)
            worksheet.conditional_format(data_start_row, excel_col, data_end_row, excel_col,
                {'type': 'formula', 'criteria': f'=AND({is_num},{diff_f}>={strong*1.5})', 'format': formats["dark_red"]})
            # 2. EMPEORA FUERTE (Rojo)
            worksheet.conditional_format(data_start_row, excel_col, data_end_row, excel_col,
                {'type': 'formula', 'criteria': f'=AND({is_num},{diff_f}>={strong},{diff_f}<{strong*1.5})', 'format': formats["red"]})
            # 3. EMPEORA LEVE (Naranja)
            worksheet.conditional_format(data_start_row, excel_col, data_end_row, excel_col,
                {'type': 'formula', 'criteria': f'=AND({is_num},{diff_f}>0,{diff_f}<{strong})', 'format': formats["orange"]})
            # 4. SIN CAMBIO (Blanco)
            worksheet.conditional_format(data_start_row, excel_col, data_end_row, excel_col,
                {'type': 'formula', 'criteria': f'=AND({is_num},ROUND({diff_f},4)=0)', 'format': formats["white"]})
            # 5. MEJORA LEVE (Verde claro)
            worksheet.conditional_format(data_start_row, excel_col, data_end_row, excel_col,
                {'type': 'formula', 'criteria': f'=AND({is_num},{diff_f}<0,{diff_f}>-{slight})', 'format': formats["very_light_green"]})
            # 6. MEJORA MODERADA (Verde)
            worksheet.conditional_format(data_start_row, excel_col, data_end_row, excel_col,
                {'type': 'formula', 'criteria': f'=AND({is_num},{diff_f}<=-{slight},{diff_f}>-{strong})', 'format': formats["green"]})
            # 7. MEJORA FUERTE (Verde Oscuro)
            worksheet.conditional_format(data_start_row, excel_col, data_end_row, excel_col,
                {'type': 'formula', 'criteria': f'=AND({is_num},{diff_f}<=-{strong})', 'format': formats["dark_green"]})
        else:
            # Columnas CMIP5: Solo bordes
            worksheet.conditional_format(data_start_row, excel_col, data_end_row, excel_col,
                {'type': 'no_blanks', 'format': formats["border"]})

    worksheet.set_column(0, 0, 18)
    worksheet.set_column(1, len(df_out.columns), 12)

# ... (todo el código anterior de procesamiento y exportación a Excel se mantiene igual)

# ============================================================
# CSV EXPORT & FINAL DEBUG PRINT
# ============================================================
df_out.to_csv(output_csv)

print("-" * 40)
print("Análisis completado exitosamente.")
print(f"Rango de Excel: Fila 3 hasta {data_end_row + 1}")
print("-" * 40)
print(f"Umbrales calculados para MAE:")
print(f"  > Leve (slight): {mae_slight:.4f}")
print(f"  > Fuerte (strong): {mae_strong:.4f}")
print("-" * 40)
print(f"Umbrales calculados para STD:")
print(f"  > Leve (slight): {std_slight:.4f}")
print(f"  > Fuerte (strong): {std_strong:.4f}")
print("-" * 40)
print(f"Archivos generados:\n  1. {output_excel}\n  2. {output_csv}")

----------------------------------------
Análisis completado exitosamente.
Rango de Excel: Fila 3 hasta 11
----------------------------------------
Umbrales calculados para MAE:
  > Leve (slight): 0.1275
  > Fuerte (strong): 0.4750
----------------------------------------
Umbrales calculados para STD:
  > Leve (slight): 0.0975
  > Fuerte (strong): 0.2975
----------------------------------------
Archivos generados:
  1. PRUDENCE_tas95_1989-2008_summary_results_analysis.xlsx
  2. PRUDENCE_tas95_1989-2008_summary_results_analysis.csv


In [4]:
import pandas as pd
import numpy as np
import re
from xlsxwriter.utility import xl_col_to_name

# ============================================================
# USER PARAMETERS
# ============================================================
input_csv = f"{areas}_{variable}_1989-2008_summary_results.csv"
output_excel = f"{areas}_{variable}_1989-2008_summary_results_abs_diff.xlsx"
output_csv = f"{areas}_{variable}_1989-2008_summary_results_abs_diff.csv"

# ============================================================
# DATA PROCESSING
# ============================================================
df = pd.read_csv(input_csv, header=[0, 1])

def split_values(val):
    """Parse 'MAE(STD)' formatted strings into numeric values."""
    if pd.isna(val) or val == "" or str(val).strip() == "0":
        return pd.Series([None, None])
    match = re.match(r"\s*([0-9.]+)\s*\(\s*([0-9.]+)\s*\)", str(val))
    if match:
        v1 = float(match.group(1))
        v2 = float(match.group(2))
        return pd.Series([v1, v2])
    return pd.Series([None, None])

index_col = df.iloc[:, 0].astype(str).str.strip()
data_dict = {}

for col in df.columns[1:]:
    season, project = col
    mae_vals, std_vals = zip(*df[col].apply(split_values).values)
    data_dict[(f"{season}_MAE", project)] = mae_vals
    data_dict[(f"{season}_STD", project)] = std_vals

df_out = pd.DataFrame(data_dict, index=index_col)
df_out = df_out.dropna(how="all")
df_out.index.name = None

# ============================================================
# COMPUTE GLOBAL ABSOLUTE DIFFERENCE DISTRIBUTION
# ============================================================
# Calculamos umbrales separados para MAE y STD para mayor precisión
def compute_abs_thresholds(metric_keyword):
    abs_diff_list = []
    cols = [c for c in df_out.columns if metric_keyword in c[0]]
    # Bloques de 2: CMIP5 es i, CMIP6 es i+1 dentro de la misma métrica
    for i in range(0, len(cols), 2):
        if i + 1 < len(cols):
            cmip5 = df_out[cols[i]]
            cmip6 = df_out[cols[i+1]]
            diff = np.abs(cmip6 - cmip5).dropna()
            abs_diff_list.extend(diff.values)
    
    if not abs_diff_list: return 0.05, 0.1
    
    slight = max(np.percentile(abs_diff_list, 25), 0.05)
    strong = max(np.percentile(abs_diff_list, 75), 0.1)
    return slight, strong

mae_slight, mae_strong = compute_abs_thresholds("MAE")
std_slight, std_strong = compute_abs_thresholds("STD")

# ============================================================
# EXCEL EXPORT - ABSOLUTE DIFFERENCE (WHITE TO RED)
# ============================================================
with pd.ExcelWriter(output_excel, engine="xlsxwriter") as writer:
    df_out.to_excel(writer, sheet_name="Analysis", startrow=2, header=False)
    workbook = writer.book
    worksheet = writer.sheets["Analysis"]

    # --- FORMATOS ---
    header_format = workbook.add_format({'bold': True, 'border': 1, 'align': 'center', 'valign': 'vcenter', 'bg_color': '#F2F2F2'})
    
    formats = {
        "white":    workbook.add_format({'bg_color': '#FFFFFF', 'border': 1, 'align': 'center'}),
        "pink":     workbook.add_format({'bg_color': '#FFEBEE', 'border': 1, 'align': 'center'}),
        "orange":   workbook.add_format({'bg_color': '#FDC08A', 'border': 1, 'align': 'center'}),
        "red":      workbook.add_format({'bg_color': '#F8696B', 'border': 1, 'align': 'center'}),
        "dark_red": workbook.add_format({'bg_color': '#8B0000', 'font_color': '#FFFFFF', 'border': 1, 'align': 'center'}),
        "border":   workbook.add_format({'border': 1, 'align': 'center'})
    }

    # --- ESCRIBIR CABECERAS ---
    worksheet.merge_range(0, 0, 1, 0, "Project", header_format)
    for i, col in enumerate(df_out.columns):
        worksheet.write(0, i + 1, col[0], header_format)
        worksheet.write(1, i + 1, col[1], header_format)

    # --- FORMATO CONDICIONAL ---
    data_start_row = 2 
    data_end_row = data_start_row + len(df_out) # Fila extra incluida

    for i, col in enumerate(df_out.columns):
        excel_col = i + 1
        
        if "CMIP6" in col[1]:
            # El CMIP5 está 2 columnas a la izquierda (MAE5-STD5-MAE6-STD6)
            cmip6_letter = xl_col_to_name(excel_col)
            cmip5_letter = xl_col_to_name(excel_col - 2)
            
            # Usar umbrales según métrica
            slight, strong = (mae_slight, mae_strong) if "MAE" in col[0] else (std_slight, std_strong)
            
            abs_diff = f"ABS({cmip6_letter}3-{cmip5_letter}3)"
            is_num = f"AND(ISNUMBER({cmip6_letter}3),ISNUMBER({cmip5_letter}3))"

            # 1. DIFERENCIA EXTREMA (Rojo Oscuro)
            worksheet.conditional_format(data_start_row, excel_col, data_end_row, excel_col,
                {'type': 'formula', 'criteria': f'=AND({is_num}, {abs_diff}>={strong*1.5})', 'format': formats["dark_red"]})
            
            # 2. DIFERENCIA FUERTE (Rojo)
            worksheet.conditional_format(data_start_row, excel_col, data_end_row, excel_col,
                {'type': 'formula', 'criteria': f'=AND({is_num}, {abs_diff}>={strong}, {abs_diff}<{strong*1.5})', 'format': formats["red"]})
            
            # 3. DIFERENCIA MODERADA (Naranja)
            worksheet.conditional_format(data_start_row, excel_col, data_end_row, excel_col,
                {'type': 'formula', 'criteria': f'=AND({is_num}, {abs_diff}>={slight}, {abs_diff}<{strong})', 'format': formats["orange"]})
            
            # 4. DIFERENCIA LEVE (Rosa)
            worksheet.conditional_format(data_start_row, excel_col, data_end_row, excel_col,
                {'type': 'formula', 'criteria': f'=AND({is_num}, {abs_diff}>0, {abs_diff}<{slight})', 'format': formats["pink"]})
            
            # 5. SIN DIFERENCIA (Blanco)
            worksheet.conditional_format(data_start_row, excel_col, data_end_row, excel_col,
                {'type': 'formula', 'criteria': f'=AND({is_num}, ROUND({abs_diff},4)==0)', 'format': formats["white"]})
        else:
            worksheet.conditional_format(data_start_row, excel_col, data_end_row, excel_col,
                {'type': 'no_blanks', 'format': formats["border"]})

    worksheet.set_column(0, 0, 18)
    worksheet.set_column(1, len(df_out.columns), 12)

# ============================================================
# EXPORT & PRINT SUMMARY
# ============================================================
df_out.to_csv(output_csv)

print("-" * 50)
print("-" * 50)
print(f"MAE Abs Thresholds: Slight={mae_slight:.4f}, Strong={mae_strong:.4f}")
print(f"STD Abs Thresholds: Slight={std_slight:.4f}, Strong={std_strong:.4f}")
print("-" * 50)
print(f"Excel generado: {output_excel}")
print(f"CSV generado:   {output_csv}")
print("-" * 50)

--------------------------------------------------
PROCESO FINALIZADO EXITOSAMENTE (MODO VALOR ABSOLUTO)
--------------------------------------------------
MAE Abs Thresholds: Slight=0.1275, Strong=0.4750
STD Abs Thresholds: Slight=0.0975, Strong=0.2950
--------------------------------------------------
Excel generado: PRUDENCE_tas95_1989-2008_summary_results_abs_diff.xlsx
CSV generado:   PRUDENCE_tas95_1989-2008_summary_results_abs_diff.csv
--------------------------------------------------
